In [58]:
import pandas as pd

churn_df = pd.read_pickle("Output/churn_df.pkl")
future_features = pd.read_pickle("Output/future_features.pkl")
future_clv_df = pd.read_pickle("Output/future_clv_predictions.pkl")

In [59]:
master_df = pd.read_pickle("Dataset/Cleaned_data/master_df.pkl")
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 36 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CustomerID             52924 non-null  str           
 1   Transaction_ID         52924 non-null  str           
 2   Transaction_Date       52924 non-null  datetime64[us]
 3   Month                  52924 non-null  str           
 4   Date                   52924 non-null  object        
 5   Week                   52924 non-null  str           
 6   Product_SKU            52924 non-null  str           
 7   Product_Description    52924 non-null  str           
 8   Product_Category       52924 non-null  str           
 9   ABC                    52924 non-null  str           
 10  Quantity               52924 non-null  int64         
 11  Avg_Price              52924 non-null  float64       
 12  Delivery_Charges       52924 non-null  float64       
 13  R

In [60]:
# ensure CustomerID columns have the same dtype
print(churn_df['CustomerID'].dtype, future_features['CustomerID'].dtype)

churn_df['CustomerID'] = churn_df['CustomerID'].astype(str)
future_features['CustomerID'] = future_features['CustomerID'].astype(str)

str int64


In [61]:
churn_df.head()

,CustomerID,frequency_cal,monetary_sum,total_quantity,unique_categories,avg_discount,coupon_usage_rate,total_delivery_fee,total_offline,total_online,...,Location_Washington Dc,KMeans_Label_1,KMeans_Label_2,KMeans_Label_3,future_orders,Churn,Churn_Probability,Churn_Segment,Revenue_At_Risk,Retention_Priority
98,12748,34,78655.491,4869,17,0.068633,0.362590,8510.90,2059500,1182404.71,...,False,False,False,True,0.0,1,0.770,Medium Risk,60564.72807,72677.673684
586,15311,16,63840.276,3598,16,0.063876,0.330144,3505.65,1194500,845752.98,...,False,False,False,True,91.0,0,0.510,Medium Risk,32558.54076,39070.248912
1113,17850,10,36683.620,1133,15,0.030303,0.306397,3162.62,793000,390789.28,...,False,False,False,True,0.0,1,0.875,High Risk,32098.16750,32098.167500
453,14667,7,27112.247,1573,13,0.056548,0.351190,1334.58,521500,278569.15,...,False,False,False,False,0.0,1,0.820,High Risk,22232.04254,22232.042540
1110,17841,16,40260.238,2259,16,0.069439,0.345114,6945.23,1244500,864774.75,...,False,False,False,True,47.0,0,0.435,Medium Risk,17513.20353,21015.844236


In [62]:
# Build customer intelligence table from churn + CLV outputs

customer_table = churn_df[[
    'CustomerID',
    'Churn_Segment',
    'Churn_Probability',
    'frequency_cal',
    'monetary_sum'
]].copy()

customer_table['CustomerID'] = customer_table['CustomerID'].astype(int)

customer_table = customer_table.merge(
    future_clv_df,
    on='CustomerID',
    how='left'
)

customer_table = customer_table.rename(columns={
    'CustomerID': 'Customer_ID',
    'Churn_Segment': 'Segment',
    'frequency_cal': 'Frequency',
    'monetary_sum': 'Monetary'
})

customer_table = customer_table[
    ['Customer_ID', 'Segment', 'Predicted_CLV',
     'Churn_Probability', 'Frequency', 'Monetary']
]

customer_table.head()

,Customer_ID,Segment,Predicted_CLV,Churn_Probability,Frequency,Monetary
0,12748,Medium Risk,3753.424585,0.770,34,78655.491
1,15311,Medium Risk,4379.451142,0.510,16,63840.276
2,17850,High Risk,1969.182284,0.875,10,36683.620
3,14667,High Risk,1492.683331,0.820,7,27112.247
4,17841,Medium Risk,4376.516371,0.435,16,40260.238


### Revenue at Risk Analysis

If anny customer has high CLV  and high churn probability  → Revenue at Risk

Current customer churn behavior puts approximately ${...} in future revenue at risk.

In [63]:
# =========================================================
# REVENUE AT RISK
# =========================================================

customer_table["Revenue_At_Risk"] = (
    customer_table["Predicted_CLV"] *
    customer_table["Churn_Probability"]
)

total_revenue_risk = customer_table["Revenue_At_Risk"].sum()

print(
    f"Estimated Revenue At Risk: "
    f"${total_revenue_risk:,.2f}"
)

Estimated Revenue At Risk: $564,640.38


## Strategic Customer Classification

Segmentation: 4 business groups.

In [64]:
# CLV threshold for high clv customers 
clv_threshold = customer_table["Predicted_CLV"].median()
print(f"CLV threshold for top customers: ${clv_threshold:.2f}")

CLV threshold for top customers: $396.81


In [65]:
# =========================================================
# CUSTOMER STRATEGY CLASSIFICATION
# =========================================================

def assign_strategy(row):
    
    segment = row["Segment"]
    revenue_at_risk = row["Revenue_At_Risk"]
    clv = row["Predicted_CLV"]
    
    # Prioritize by segment + value
    if segment == "High Risk":
        if clv >= clv_threshold:
            return "Emergency Intervention"
        else:
            return "High-Risk Automation"
    
    elif segment == "Medium Risk":
        if clv >= clv_threshold:
            return "Critical Retention"
        else:
            return "Targeted Campaign"
    
    else:  # Safe Customer
        if clv >= clv_threshold:
            return "Loyal Growth"
        else:
            return "Nurture"


customer_table["Strategy_Group"] = customer_table.apply(
    assign_strategy,
    axis=1
)

In [66]:
display(customer_table.head())
display(customer_table.describe().T)

,Customer_ID,Segment,Predicted_CLV,Churn_Probability,Frequency,Monetary,Revenue_At_Risk,Strategy_Group
0,12748,Medium Risk,3753.424585,0.770,34,78655.491,2890.136930,Critical Retention
1,15311,Medium Risk,4379.451142,0.510,16,63840.276,2233.520082,Critical Retention
2,17850,High Risk,1969.182284,0.875,10,36683.620,1723.034499,Emergency Intervention
3,14667,High Risk,1492.683331,0.820,7,27112.247,1224.000331,Emergency Intervention
4,17841,Medium Risk,4376.516371,0.435,16,40260.238,1903.784621,Critical Retention


,count,mean,std,min,25%,50%,75%,max
Customer_ID,1211.0,15364.936416,1748.337393,12346.00,13881.5000,15380.000000,16898.500000,18283.000000
Predicted_CLV,1211.0,741.780646,839.782540,0.00,0.0000,396.811276,1430.467268,5046.976425
Churn_Probability,1211.0,0.761321,0.328689,0.01,0.6475,0.935000,0.990000,1.000000
Frequency,1211.0,2.020644,1.944368,1.00,1.0000,1.000000,2.000000,34.000000
Monetary,1211.0,2875.822228,4720.837507,7.20,579.8905,1581.107000,3474.901000,78655.491000
Revenue_At_Risk,1211.0,466.259604,633.538354,0.00,0.0000,54.665781,1040.296100,2890.136930


In [67]:
customer_table.to_pickle("Output/customer_table.pkl")

## RISK ANALYSIS FOR EMERGENCY INTERVENTION GROUP

In [68]:
#=========================================================
# REVENUE AT RISK ANALYSIS
#=========================================================  

print("-" * 50)
print(f"Proportion of Emergency Intervention Group)")
print("-" * 50)
print(f"Total Revenue at Risk: ${total_risk_revenue:,.2f}")
print(f"Revenue at Risk for Emergency Intervention: ${top_risk:,.2f}")
print(f"Percentage of Total Revenue at Risk for Emergency Intervention: {top_risk/total_risk_revenue:.2%}")
print("-" * 50)

--------------------------------------------------
Proportion of Emergency Intervention Group)
--------------------------------------------------
Total Revenue at Risk: $564,640.38
Revenue at Risk for Emergency Intervention: $418,514.14
Percentage of Total Revenue at Risk for Emergency Intervention: 74.12%
--------------------------------------------------


In [69]:
baseline_retention = 1 - customer_table[customer_table['Strategy_Group']=='Emergency Intervention']['Churn_Probability'].mean()
min_retention = 1 - customer_table[customer_table['Strategy_Group']=='Emergency Intervention']['Churn_Probability'].max()
max_retention = 1 - customer_table[customer_table['Strategy_Group']=='Emergency Intervention']['Churn_Probability'].min()
print(f"Minimum Retention Rate for Emergency Intervention group: {min_retention:.2%}")
print(f"Maximum Retention Rate for Emergency Intervention group: {max_retention:.2%}")
print(f"Baseline Retention Rate for Emergency Intervention group: {baseline_retention:.2%}")

Minimum Retention Rate for Emergency Intervention group: 0.50%
Maximum Retention Rate for Emergency Intervention group: 20.00%
Baseline Retention Rate for Emergency Intervention group: 10.75%


In [70]:
#=========================================================
# RETENTION LIFT TARGETS & EXPECTED RECOVERY
#=========================================================

# Goal: Increase retention in Emergency Intervention group to at least 25% (from baseline of ~10%)
lift_target_min = 0.25 - baseline_retention

lift_target_max = 0.25 - min_retention

recovery_min = top_risk * lift_target_min
recovery_max = top_risk * lift_target_max

print("-" * 50)
print(f"Expected Retention to Value in the next 90 days (for a specific group)")
print("-" * 50)
print(f"Group Revenue at Risk: ${top_risk:,.2f}")
print(f"Retention Baseline:      {baseline_retention:.2%}")
print(f"Target Lift (KPI):      +{lift_target_min:.1%} to +{lift_target_max:.1%}")
print("-" * 50)
print(f"EXPECTED REVENUE RECOVERY: ${recovery_min:,.0f} - ${recovery_max:,.0f} / quarter")
print("-" * 50)

--------------------------------------------------
Expected Retention to Value in the next 90 days (for a specific group)
--------------------------------------------------
Group Revenue at Risk: $418,514.14
Retention Baseline:      10.75%
Target Lift (KPI):      +14.3% to +24.5%
--------------------------------------------------
EXPECTED REVENUE RECOVERY: $59,656 - $102,536 / quarter
--------------------------------------------------
